# 🧪 ADMET Toxicity Prediction with ChemBERTa
## Milestone 1: Fine-Tuning a Molecular Transformer on Tox21

**What we're building:** A classifier that accepts a SMILES string (a text-based representation of a molecule) and predicts whether it activates the **SR-p53 stress response pathway** — a key toxicity signal used in early-stage drug safety screening.

**What you'll learn in this notebook:**
- What SMILES strings are and why transformers can read them
- How tokenization works for molecular data
- How LoRA lets you fine-tune a large model cheaply
- How to replace a model's output head for a new task
- How to evaluate a classifier with ROC-AUC (the right metric for imbalanced data)
- How to track experiments and save reproducibility metadata

**Technology stack:**
```
SMILES Input
    ↓
ChemBERTa-77M (RoBERTa pretrained on 77M molecules)
    ↓  [frozen weights + LoRA adapters]
Classification Head (Linear 600 → 2)
    ↓
TOXIC / NON-TOXIC + confidence score
```

**Runtime:** Use a free **T4 GPU** (`Runtime → Change runtime type → T4 GPU`)

---

## Section 1: Setup

In [1]:
# Install all dependencies
# - transformers: HuggingFace model library
# - datasets: HuggingFace dataset library
# - peft: Parameter-Efficient Fine-Tuning (LoRA lives here)
# - accelerate: Backend that makes distributed/GPU training work seamlessly
# - wandb: Experiment tracking
# - scikit-learn: Evaluation metrics (ROC-AUC, F1, etc.)
# - python-dotenv: Loads environment variables from a .env file (local dev)

!pip install -q transformers datasets peft accelerate wandb scikit-learn huggingface_hub python-dotenv


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import json
import math
import numpy as np
import pandas as pd
import torch
from datetime import datetime

from datasets import load_dataset, Dataset as HFDataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
import wandb
from huggingface_hub import login

# Reproducibility — set a global seed so results are consistent across runs
SEED = 42
set_seed(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/Users/michaelmalloy/.pyenv/versions/3.12.4/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.12.0
CUDA available: False


### Configuration

All hyperparameters are defined here in one place — this is important for reproducibility and governance. When you save this notebook alongside a trained model, someone can look here and know exactly how the model was produced.

**The 12 Tox21 target options** (change `TARGET_TASK` to try others):
```
Nuclear receptors:  NR-AR, NR-AR-LBD, NR-AhR, NR-Aromatase, NR-ER, NR-ER-LBD, NR-PPAR-gamma
Stress response:    SR-ARE, SR-ATAD5, SR-HSE, SR-MMP, SR-p53
```

In [4]:
# ── Model & Data ──────────────────────────────────────────────────────────────
BASE_MODEL   = "DeepChem/ChemBERTa-77M-MTR"          # Pretrained molecular transformer
DATASET_NAME = "scikit-fingerprints/MoleculeNet_Tox21"
TARGET_TASK  = "SR-p53"   # <-- Change this to try other Tox21 targets

PROJECT_NAME = "admet-toxicity"
HF_USER      = "mike-malloy"   # <-- Replace with your HuggingFace username

# ── LoRA Hyperparameters ──────────────────────────────────────────────────────
# r (rank): The bottleneck dimension of the LoRA matrices.
#   Higher = more expressive, more parameters. 8–32 is typical for small models.
LORA_R           = 16
# lora_alpha: Scaling factor. Rule of thumb: set to 2× rank.
#   Controls how strongly the LoRA updates influence the output.
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.1
# Which attention matrices to adapt. "query" and "value" is a common minimal choice.
# Adding "key" and "dense" gives more capacity at the cost of more parameters.
TARGET_MODULES   = ["query", "value"]

# ── Training Hyperparameters ──────────────────────────────────────────────────
EPOCHS          = 10
BATCH_SIZE      = 32
LEARNING_RATE   = 2e-4
WEIGHT_DECAY    = 0.01
WARMUP_RATIO    = 0.1     # Fraction of training steps used for learning rate warmup
MAX_SEQ_LENGTH  = 128     # Max token length; most SMILES are well under this

# ── Logging & Checkpointing ───────────────────────────────────────────────────
LOG_STEPS  = 10
EVAL_STEPS = 50
SAVE_STEPS = 50
EARLY_STOPPING_PATIENCE = 4   # Stop if val ROC-AUC doesn't improve for 4 evals

# ── Derived Names ─────────────────────────────────────────────────────────────
RUN_NAME       = f"chemberta-tox21-{TARGET_TASK.lower()}-{datetime.now():%Y%m%d_%H%M}"
HUB_MODEL_NAME = f"{HF_USER}/{RUN_NAME}"

print(f"Model:      {BASE_MODEL}")
print(f"Dataset:    {DATASET_NAME}")
print(f"Target:     {TARGET_TASK}")
print(f"Run name:   {RUN_NAME}")

Model:      DeepChem/ChemBERTa-77M-MTR
Dataset:    scikit-fingerprints/MoleculeNet_Tox21
Target:     SR-p53
Run name:   chemberta-tox21-sr-p53-20260519_0805


### Login to HuggingFace and Weights & Biases

1. Click the **🔑 key icon** in the Colab left sidebar
2. Add secret `HF_TOKEN` → your token from https://huggingface.co/settings/tokens
3. Add secret `WANDB_API_KEY` → your key from https://wandb.ai/settings

In [5]:
# ── Detect environment and load secrets ───────────────────────────────────────
# Uses Colab secrets when running in the Colab UI,
# or a .env file when running locally (Cursor, VS Code, terminal).

def _get_secret(key: str) -> str:
    """Return secret from Colab userdata (Colab UI) or .env file (local)."""
    try:
        from google.colab import userdata
        return userdata.get(key)
    except (ImportError, Exception):
        # Not in Colab (or Colab secrets unavailable) — load from .env file.
        # find_dotenv() walks up from CWD to locate the .env, so it works
        # regardless of what directory Jupyter started in.
        try:
            from dotenv import load_dotenv, find_dotenv
            load_dotenv(find_dotenv())
        except ImportError:
            pass  # python-dotenv not installed; rely on variables already in env
        val = os.environ.get(key)
        if not val:
            raise EnvironmentError(
                f"Secret '{key}' not found. "
                f"Add {key}=<your-value> to a .env file in the project directory."
            )
        return val

# HuggingFace login — needed to push your fine-tuned model to the Hub
hf_token = _get_secret('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

# Weights & Biases login — tracks your training curves, metrics, and hyperparameters
wandb_api_key = _get_secret('WANDB_API_KEY')
os.environ["WANDB_API_KEY"]   = wandb_api_key
os.environ["WANDB_PROJECT"]   = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"]     = "false"
wandb.login()

# Initialize a WandB run — log hyperparameters so each run is fully documented
wandb.init(
    project=PROJECT_NAME,
    name=RUN_NAME,
    config={
        "base_model":     BASE_MODEL,
        "dataset":        DATASET_NAME,
        "target_task":    TARGET_TASK,
        "lora_r":         LORA_R,
        "lora_alpha":     LORA_ALPHA,
        "lora_dropout":   LORA_DROPOUT,
        "target_modules": TARGET_MODULES,
        "epochs":         EPOCHS,
        "batch_size":     BATCH_SIZE,
        "learning_rate":  LEARNING_RATE,
        "seed":           SEED,
    }
)

print("✅ Logged in to HuggingFace and Weights & Biases")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: mike-malloy-2004 (mike-malloy) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Logged in to HuggingFace and Weights & Biases


---
## Section 2: Data

### What is a SMILES string?

**SMILES** (Simplified Molecular Input Line Entry System) is a notation that encodes a molecule's structure as a text string. Transformers can read SMILES the same way they read sentences — because both are sequences of tokens with grammar-like rules.

| Molecule | SMILES | What it means |
|----------|--------|---------------|
| Water | `O` | One oxygen atom |
| Ethanol | `CCO` | Two carbons, one oxygen |
| Aspirin | `CC(=O)Oc1ccccc1C(=O)O` | Acetyl group + benzene ring + carboxylic acid |
| Caffeine | `Cn1c(=O)c2c(ncn2C)n(c1=O)C` | Fused ring system with nitrogen atoms |

### What is Tox21?

The **Toxicology in the 21st Century (Tox21)** dataset contains toxicity measurements for ~8,000 compounds across **12 biological targets**. For each compound+target pair, the label is:
- **1 (Active/Toxic):** The compound triggered the biological response
- **0 (Inactive/Non-toxic):** No response observed
- **NaN:** The compound was not tested against this target

We're predicting **SR-p53**: whether a compound activates the p53 tumor suppressor stress response pathway. p53 activation indicates the compound may be causing DNA damage — a major drug safety red flag.

In [10]:
# Load the dataset from HuggingFace Hub
# This will download train/validation/test splits automatically
print("Loading Tox21 dataset...")
raw_dataset = load_dataset(DATASET_NAME)
print(raw_dataset)

# Inspect the feature columns
print("\nColumns:", list(raw_dataset['train'].features.keys()))
print("\nFirst example:")
print(raw_dataset['train'][0])

Loading Tox21 dataset...
DatasetDict({
    train: Dataset({
        features: ['SMILES', 'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53'],
        num_rows: 7831
    })
})

Columns: ['SMILES', 'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']

First example:
{'SMILES': 'CCOc1ccc2nc(S(N)(=O)=O)sc2c1', 'NR-AR': 0.0, 'NR-AR-LBD': 0.0, 'NR-AhR': 1.0, 'NR-Aromatase': None, 'NR-ER': None, 'NR-ER-LBD': 0.0, 'NR-PPAR-gamma': 0.0, 'SR-ARE': 1.0, 'SR-ATAD5': 0.0, 'SR-HSE': 0.0, 'SR-MMP': 0.0, 'SR-p53': 0.0}


In [11]:
# ── Auto-detect column names ──────────────────────────────────────────────────
sample = raw_dataset['train'][0]
SMILES_COL = 'smiles' if 'smiles' in sample else 'SMILES'

# All task columns = everything that isn't the SMILES or metadata column
NON_TASK_COLS = {SMILES_COL.lower(), 'smiles', 'mol_id', 'id',
                 'selfies', '__index_level_0__'}
TASK_COLUMNS = [c for c in raw_dataset['train'].features.keys()
                if c.lower() not in NON_TASK_COLS]

print(f"SMILES column:   '{SMILES_COL}'")
print(f"Available splits: {list(raw_dataset.keys())}")
print(f"Tox21 targets ({len(TASK_COLUMNS)}): {TASK_COLUMNS}")

# Verify our chosen target exists
assert TARGET_TASK in TASK_COLUMNS, (
    f"Target '{TARGET_TASK}' not found. Available: {TASK_COLUMNS}"
)
print(f"\n✅ Target '{TARGET_TASK}' confirmed.")
print("ℹ️  No pre-built val/test splits — will create 70/15/15 splits from train.")

SMILES column:   'SMILES'
Available splits: ['train']
Tox21 targets (12): ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']

✅ Target 'SR-p53' confirmed.
ℹ️  No pre-built val/test splits — will create 70/15/15 splits from train.


In [12]:
# ── Explore label distribution across all targets ─────────────────────────────
# This reveals class imbalance — important for choosing the right metric.
# Most Tox21 targets are heavily imbalanced (many more non-toxic than toxic).

df_train_raw = raw_dataset['train'].to_pandas()

print(f"{'Target':<20} {'N (tested)':<12} {'N Toxic':<10} {'% Toxic':<10}")
print("-" * 52)
for t in TASK_COLUMNS:
    col = df_train_raw[t]
    n_tested = col.notna().sum()
    n_toxic  = (col == 1).sum()
    pct      = 100 * n_toxic / n_tested if n_tested > 0 else 0
    marker   = " ← selected" if t == TARGET_TASK else ""
    print(f"{t:<20} {n_tested:<12} {n_toxic:<10} {pct:<10.1f}%{marker}")

Target               N (tested)   N Toxic    % Toxic   
----------------------------------------------------
NR-AR                7265         309        4.3       %
NR-AR-LBD            6758         237        3.5       %
NR-AhR               6549         768        11.7      %
NR-Aromatase         5821         300        5.2       %
NR-ER                6193         793        12.8      %
NR-ER-LBD            6955         350        5.0       %
NR-PPAR-gamma        6450         186        2.9       %
SR-ARE               5832         942        16.2      %
SR-ATAD5             7072         264        3.7       %
SR-HSE               6467         372        5.8       %
SR-MMP               5810         918        15.8      %
SR-p53               6774         423        6.2       % ← selected


In [13]:
# ── Preprocess and split ──────────────────────────────────────────────────────
# The dataset only has a 'train' split, so we create val and test splits manually.
# stratify=df['label'] preserves the toxic/non-toxic ratio in each split.

from sklearn.model_selection import train_test_split

df = raw_dataset['train'].to_pandas()
df = df[df[TARGET_TASK].notna()].copy()
df['label'] = df[TARGET_TASK].astype(int)
df = df[[SMILES_COL, 'label']].rename(columns={SMILES_COL: 'smiles'}).reset_index(drop=True)

# 70% train, 15% val, 15% test — all stratified to preserve class balance
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df['label'])
val_df,  test_df  = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['label'])

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    pos_rate = df['label'].mean()
    print(f"{name:<6}: {len(df):>5} samples | "
          f"{df['label'].sum():>4} toxic ({pos_rate:.1%}) | "
          f"{(df['label']==0).sum():>4} non-toxic ({1-pos_rate:.1%})")

print("\nSample molecules:")
for _, row in train_df.sample(5, random_state=SEED).iterrows():
    label_str = "TOXIC    " if row['label'] == 1 else "non-toxic"
    print(f"  [{label_str}] {row['smiles']}")

Train :  4741 samples |  296 toxic (6.2%) | 4445 non-toxic (93.8%)
Val   :  1016 samples |   63 toxic (6.2%) |  953 non-toxic (93.8%)
Test  :  1017 samples |   64 toxic (6.3%) |  953 non-toxic (93.7%)

Sample molecules:
  [non-toxic] OC[C@H]1O[C@@H]2O[C@@H]3[C@@H](CO)O[C@H](O[C@@H]4[C@@H](CO)O[C@H](O[C@@H]5[C@@H](CO)O[C@H](O[C@@H]6[C@@H](CO)O[C@H](O[C@@H]7[C@@H](CO)O[C@H](O[C@H]1[C@H](O)[C@H]2O)[C@H](O)[C@H]7O)[C@H](O)[C@H]6O)[C@H](O)[C@H]5O)[C@H](O)[C@H]4O)[C@H](O)[C@H]3O
  [non-toxic] Nc1ccc(Sc2ccc(N)cc2)cc1
  [non-toxic] Oc1ccc(Cl)cc1Cc1ccccc1
  [non-toxic] CC(C)(O)c1ccc(CNC(=O)c2cccnc2Oc2ccc3nonc3c2)cc1
  [non-toxic] O=[Si]=O


---
## Section 3: Tokenization

### What does a tokenizer do?

A tokenizer converts raw text into a sequence of integer IDs that the model can process. ChemBERTa uses a **custom SMILES vocabulary** of ~591 tokens covering atoms, bonds, branches, rings, and special symbols.

Three special tokens are always added:
- **`[CLS]`** — prepended to every sequence. The transformer's final representation of this token is used as the "summary" of the whole molecule — this is what gets fed into the classification head.
- **`[SEP]`** — appended to mark the end of the sequence.
- **`[PAD]`** — used to make all sequences the same length within a batch.

```
"CC(=O)O"  →  [CLS, C, C, (, =, O, ), O, SEP, PAD, PAD, ...]
                ↑                                    ↑
          used for classification             ignored by model
```

In [14]:
# Load the ChemBERTa tokenizer
# trust_remote_code=True is needed because ChemBERTa uses a custom tokenizer class
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

print(f"Vocabulary size:    {tokenizer.vocab_size} tokens")
print(f"Max model length:   {tokenizer.model_max_length}")
print(f"Special tokens:     CLS='{tokenizer.cls_token}', "
      f"SEP='{tokenizer.sep_token}', PAD='{tokenizer.pad_token}'")

# ── See tokenization in action ─────────────────────────────────────────────────
demo_molecules = {
    "Acetic acid (simple)": "CC(=O)O",
    "Aspirin (medium)": "CC(=O)Oc1ccccc1C(=O)O",
    "Caffeine (complex)": "Cn1c(=O)c2c(ncn2C)n(c1=O)C",
}

print("\n" + "=" * 60)
for name, smi in demo_molecules.items():
    enc = tokenizer(smi)
    tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'])
    print(f"\n{name}")
    print(f"  SMILES: {smi}")
    print(f"  Tokens: {tokens}")
    print(f"  IDs:    {enc['input_ids']}")
    print(f"  Length: {len(enc['input_ids'])} tokens")

Vocabulary size:    591 tokens
Max model length:   512
Special tokens:     CLS='[CLS]', SEP='[SEP]', PAD='[PAD]'


Acetic acid (simple)
  SMILES: CC(=O)O
  Tokens: ['[CLS]', 'C', 'C', '(', '=', 'O', ')', 'O', '[SEP]']
  IDs:    [12, 16, 16, 17, 22, 19, 18, 19, 13]
  Length: 9 tokens

Aspirin (medium)
  SMILES: CC(=O)Oc1ccccc1C(=O)O
  Tokens: ['[CLS]', 'C', 'C', '(', '=', 'O', ')', 'O', 'c', '1', 'c', 'c', 'c', 'c', 'c', '1', 'C', '(', '=', 'O', ')', 'O', '[SEP]']
  IDs:    [12, 16, 16, 17, 22, 19, 18, 19, 15, 20, 15, 15, 15, 15, 15, 20, 16, 17, 22, 19, 18, 19, 13]
  Length: 23 tokens

Caffeine (complex)
  SMILES: Cn1c(=O)c2c(ncn2C)n(c1=O)C
  Tokens: ['[CLS]', 'C', 'n', '1', 'c', '(', '=', 'O', ')', 'c', '2', 'c', '(', 'n', 'c', 'n', '2', 'C', ')', 'n', '(', 'c', '1', '=', 'O', ')', 'C', '[SEP]']
  IDs:    [12, 16, 25, 20, 15, 17, 22, 19, 18, 15, 21, 15, 17, 25, 15, 25, 21, 16, 18, 25, 17, 15, 20, 22, 19, 18, 16, 13]
  Length: 28 tokens


In [15]:
# ── Check token length distribution in our dataset ────────────────────────────
# This tells us if MAX_SEQ_LENGTH=128 is sufficient (it almost always is for SMILES)

sample_smiles = train_df['smiles'].sample(500, random_state=SEED).tolist()
lengths = [len(tokenizer(s)['input_ids']) for s in sample_smiles]

print(f"Token length stats (sample of 500 training molecules):")
print(f"  Min:    {min(lengths)}")
print(f"  Max:    {max(lengths)}")
print(f"  Mean:   {np.mean(lengths):.1f}")
print(f"  Median: {np.median(lengths):.0f}")
print(f"  95th %: {np.percentile(lengths, 95):.0f}")
pct_truncated = sum(1 for l in lengths if l > MAX_SEQ_LENGTH) / len(lengths)
print(f"  % truncated at {MAX_SEQ_LENGTH}: {pct_truncated:.1%}")

Token length stats (sample of 500 training molecules):
  Min:    3
  Max:    159
  Mean:   32.2
  Median: 26
  95th %: 75
  % truncated at 128: 0.4%


In [16]:
# ── Tokenize all splits ───────────────────────────────────────────────────────
# padding='max_length' pads every sequence to MAX_SEQ_LENGTH
# truncation=True cuts any sequence longer than MAX_SEQ_LENGTH

def tokenize_batch(examples):
    return tokenizer(
        examples['smiles'],
        padding='max_length',
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )

# Convert DataFrames → HuggingFace Datasets
train_dataset = HFDataset.from_pandas(train_df)
val_dataset   = HFDataset.from_pandas(val_df)
test_dataset  = HFDataset.from_pandas(test_df)

# Apply tokenization (batched=True is much faster than row-by-row)
train_dataset = train_dataset.map(tokenize_batch, batched=True)
val_dataset   = val_dataset.map(tokenize_batch,   batched=True)
test_dataset  = test_dataset.map(tokenize_batch,  batched=True)

# Tell HuggingFace which columns are tensors the model needs
tensor_cols = ['input_ids', 'attention_mask', 'label']
train_dataset.set_format('torch', columns=tensor_cols)
val_dataset.set_format('torch',   columns=tensor_cols)
test_dataset.set_format('torch',  columns=tensor_cols)

print("Tokenized dataset:")
print(train_dataset)
print("\nSample batch keys:", list(train_dataset[0].keys()))
print("input_ids shape:  ", train_dataset[0]['input_ids'].shape)
print("label:            ", train_dataset[0]['label'])

Map: 100%|██████████| 1017/1017 [00:00<00:00, 28285.39 examples/s]

Tokenized dataset:
Dataset({
    features: ['smiles', 'label', 'input_ids', 'attention_mask'],
    num_rows: 4741
})

Sample batch keys: ['label', 'input_ids', 'attention_mask']
input_ids shape:   torch.Size([128])
label:             tensor(0)


---
## Section 4: Model — Load ChemBERTa and Apply LoRA

### Architecture overview

**ChemBERTa-77M-MTR** is a RoBERTa-style encoder transformer with:
- 6 transformer layers
- 600-dimensional hidden states
- 12 attention heads per layer
- Pre-trained on 77M SMILES from PubChem using multi-task regression (predicting 200 RDKit molecular properties)

We load it with `AutoModelForSequenceClassification`, which **replaces the original regression head** with a new 2-class linear layer: `Linear(600 → 2)`.

### What is LoRA?

**Low-Rank Adaptation (LoRA)** is a fine-tuning technique that:
1. **Freezes** all original model weights (they don't change during training)
2. **Injects** small trainable matrices into selected layers

For a weight matrix **W** of shape `(d × d)`, instead of updating all `d²` values:
```
ΔW = B × A
  where  B: (d × r),  A: (r × d),  r << d
```

With `r=16` and ChemBERTa's `d=600`:
- Full update per layer: `600 × 600 = 360,000 params`
- LoRA update per layer: `600×16 + 16×600 = 19,200 params` → **94.7% reduction**

The classification head (`Linear(600, 2)`) is **fully trainable** — it starts from scratch and needs unrestricted updates.

In [17]:
# ── Load the base model ───────────────────────────────────────────────────────
# ignore_mismatched_sizes=True: the pretrained model had a regression head;
# we're replacing it with a 2-class classification head (different shape).
# PyTorch will warn about this — it's expected and intentional.

base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: "NON-TOXIC", 1: "TOXIC"},
    label2id={"NON-TOXIC": 0, "TOXIC": 1},
    ignore_mismatched_sizes=True,
    trust_remote_code=True,
)

total_params = sum(p.numel() for p in base_model.parameters())
print(f"Total parameters:  {total_params:,}")
print(f"Memory footprint:  {base_model.get_memory_footprint() / 1e6:.1f} MB")
print("\nTop-level modules:")
for name, module in base_model.named_children():
    n_params = sum(p.numel() for p in module.parameters())
    print(f"  {name:<20} {type(module).__name__:<30} {n_params:>10,} params")

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `199`.
Loading weights: 100%|██████████| 53/53 [00:00<00:00, 53591.64it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: DeepChem/ChemBERTa-77M-MTR
Key                        | Status     | 
---------------------------+------------+-
norm_std                   | UNEXPECTED | 
regression.out_proj.weight | UNEXPECTED | 
regression.dense.bias      | UNEXPECTED | 
norm_mean                  | UNEXPECTED | 
regression.out_proj.bias   | UNEXPECTED | 
regression.dense.weight    | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider trainin

Total parameters:  3,428,210
Memory footprint:  13.7 MB

Top-level modules:
  roberta              RobertaModel                    3,279,600 params
  classifier           RobertaClassificationHead         148,610 params


In [18]:
# ── Apply LoRA ────────────────────────────────────────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,      # Tells PEFT this is a sequence classification task
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,   # Which linear layers to inject LoRA into
    bias="none",                     # Don't adapt bias terms (keeps param count low)
    modules_to_save=["classifier"],  # Fully train the new classification head
)

model = get_peft_model(base_model, lora_config)

# Print the breakdown of trainable vs frozen parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:>10,}  ({100*trainable/total:.2f}% of total)")
print(f"Frozen parameters:    {total-trainable:>10,}  ({100*(total-trainable)/total:.2f}% of total)")
print(f"Total parameters:     {total:>10,}")
print()
model.print_trainable_parameters()

Trainable parameters:    222,338  (6.09% of total)
Frozen parameters:     3,428,210  (93.91% of total)
Total parameters:      3,650,548

trainable params: 222,338 || all params: 3,650,548 || trainable%: 6.0905


In [19]:
# ── Inspect LoRA structure ────────────────────────────────────────────────────
# This shows you exactly which layers have LoRA adapters injected.
# lora_A and lora_B are the two trainable matrices; original weight is frozen.

print("Layers with LoRA adapters:")
for name, param in model.named_parameters():
    if param.requires_grad and 'lora' in name:
        print(f"  {name:<70} shape={list(param.shape)}")

Layers with LoRA adapters:
  base_model.model.roberta.encoder.layer.0.attention.self.query.lora_A.default.weight shape=[16, 384]
  base_model.model.roberta.encoder.layer.0.attention.self.query.lora_B.default.weight shape=[384, 16]
  base_model.model.roberta.encoder.layer.0.attention.self.value.lora_A.default.weight shape=[16, 384]
  base_model.model.roberta.encoder.layer.0.attention.self.value.lora_B.default.weight shape=[384, 16]
  base_model.model.roberta.encoder.layer.1.attention.self.query.lora_A.default.weight shape=[16, 384]
  base_model.model.roberta.encoder.layer.1.attention.self.query.lora_B.default.weight shape=[384, 16]
  base_model.model.roberta.encoder.layer.1.attention.self.value.lora_A.default.weight shape=[16, 384]
  base_model.model.roberta.encoder.layer.1.attention.self.value.lora_B.default.weight shape=[384, 16]
  base_model.model.roberta.encoder.layer.2.attention.self.query.lora_A.default.weight shape=[16, 384]
  base_model.model.roberta.encoder.layer.2.attention.se

---
## Section 5: Training

### Why ROC-AUC and not accuracy?

Look at the label distribution you printed earlier. Tox21 SR-p53 is typically ~10–15% toxic. If a model just predicted "non-toxic" for everything, it would achieve ~87% accuracy — but it would be completely useless.

**ROC-AUC** (Receiver Operating Characteristic — Area Under Curve) measures how well the model **ranks** toxic vs. non-toxic molecules across all possible decision thresholds. It's immune to class imbalance and is the standard metric in drug discovery toxicity prediction.
- **1.0** = perfect discrimination
- **0.5** = random guessing
- **> 0.7** = generally useful for screening

In [28]:
# ── Evaluation metrics ────────────────────────────────────────────────────────

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # Convert logits to probabilities via softmax
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()
    preds = np.argmax(logits, axis=-1)

    # Probability of being TOXIC (class index 1)
    pos_probs = probs[:, 1]

    # ROC-AUC: primary metric
    try:
        auc = roc_auc_score(labels, pos_probs)
    except ValueError:
        # Can happen if a small eval batch has only one class
        auc = 0.0

    # Supporting metrics
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, zero_division=0)

    return {
        "roc_auc":  round(auc, 4),
        "accuracy": round(acc, 4),
        "f1":       round(f1, 4),
    }

In [29]:
# ── Training Arguments ────────────────────────────────────────────────────────
# Determine precision and device support
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16
use_mps  = not torch.cuda.is_available() and torch.backends.mps.is_available()

if use_bf16:
    precision = "bf16 (CUDA)"
elif use_fp16:
    precision = "fp16 (CUDA)"
elif use_mps:
    precision = "fp32 (MPS / Apple Silicon)"
else:
    precision = "fp32 (CPU)"
print(f"Precision/device: {precision}")

training_args = TrainingArguments(
    output_dir=RUN_NAME,

    # ── Training duration
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,

    # ── Optimization
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    max_grad_norm=1.0,

    # ── Precision
    fp16=use_fp16,
    bf16=use_bf16,

    # ── Evaluation & checkpointing
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="roc_auc",
    greater_is_better=True,

    # ── Logging
    logging_steps=LOG_STEPS,
    report_to="wandb",
    run_name=RUN_NAME,

    # ── MPS: pin_memory is unsupported, disable to suppress warnings
    dataloader_pin_memory=False,

    # ── HuggingFace Hub
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    hub_strategy="every_save",

    seed=SEED,
)

print(f"Steps per epoch: {math.ceil(len(train_dataset) / BATCH_SIZE)}")
print(f"Total steps:     {math.ceil(len(train_dataset) / BATCH_SIZE) * EPOCHS}")

Precision/device: fp32 (MPS / Apple Silicon)
Steps per epoch: 149
Total steps:     1490


In [30]:
# ── Class-weighted Trainer ────────────────────────────────────────────────────
# With 93.8% non-toxic samples, a plain cross-entropy loss lets the model "cheat"
# by predicting non-toxic for everything. Class weights penalise missing a toxic
# compound proportionally to how rare it is in the training set.

from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1])
weights = compute_class_weight('balanced', classes=classes, y=train_df['label'].values)
class_weights = torch.tensor(weights, dtype=torch.float32)
print(f"Class weights — NON-TOXIC: {weights[0]:.3f}  TOXIC: {weights[1]:.3f}")
print(f"(toxic mistakes penalised ~{weights[1]/weights[0]:.0f}× more than non-toxic)")

class WeightedLossTrainer(Trainer):
    """Trainer that uses class-weighted cross-entropy to handle label imbalance."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        loss = torch.nn.functional.cross_entropy(
            logits,
            labels,
            weight=class_weights.to(logits.device),
        )
        return (loss, outputs) if return_outputs else loss

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

print(f"\nTraining on {len(train_dataset):,} samples")
print(f"Validating on {len(val_dataset):,} samples")
print(f"Early stopping patience: {EARLY_STOPPING_PATIENCE} evaluations\n")
print("Starting training... 🚀")

train_result = trainer.train()

Class weights — NON-TOXIC: 0.533  TOXIC: 8.008
(toxic mistakes penalised ~15× more than non-toxic)

Training on 4,741 samples
Validating on 1,016 samples
Early stopping patience: 4 evaluations

Starting training... 🚀


Step,Training Loss,Validation Loss,Roc Auc,Accuracy,F1
50,0.581856,0.583395,0.793400,0.901600,0.342100
100,0.589634,0.577496,0.792800,0.843500,0.287000
150,0.500337,0.589588,0.808700,0.867100,0.307700
200,0.392030,0.593722,0.804200,0.859300,0.309200
250,0.581197,0.579080,0.802200,0.830700,0.306500
300,0.411848,0.579379,0.807100,0.785400,0.268500
350,0.466017,0.621698,0.812000,0.837600,0.303800
400,0.612894,0.630853,0.813300,0.848400,0.300000
450,0.396840,0.601353,0.813200,0.822800,0.296900
500,0.676166,0.637541,0.814700,0.828700,0.304000


---
## Section 6: Evaluation

In [31]:
# ── Evaluate on held-out test set ─────────────────────────────────────────────
# IMPORTANT: We only run this ONCE after training is fully complete.
# Running it repeatedly while adjusting hyperparameters would leak test-set info.

test_results = trainer.evaluate(test_dataset)

print("\n" + "=" * 45)
print(f"  Test Results: {TARGET_TASK}")
print("=" * 45)
print(f"  ROC-AUC:  {test_results['eval_roc_auc']:.4f}  (primary metric)")
print(f"  Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"  F1 Score: {test_results['eval_f1']:.4f}")
print(f"  Loss:     {test_results['eval_loss']:.4f}")
print("=" * 45)

# Log test metrics to WandB
wandb.log({
    "test/roc_auc":  test_results['eval_roc_auc'],
    "test/accuracy": test_results['eval_accuracy'],
    "test/f1":       test_results['eval_f1'],
    "test/loss":     test_results['eval_loss'],
})

Training Loss,Validation Loss,Step,Roc Auc,Accuracy,F1
0.467998,0.575888,800,0.841900,0.867300,0.341500



  Test Results: SR-p53
  ROC-AUC:  0.8419  (primary metric)
  Accuracy: 0.8673
  F1 Score: 0.3415
  Loss:     0.5759


In [32]:
# ── Tune decision threshold on the validation set, then apply to test ─────────
# Default threshold (0.5) is wrong for imbalanced data. We find the threshold
# that maximises F1 on the validation set, then use it once on the test set.

from sklearn.metrics import classification_report, confusion_matrix, f1_score as f1

# Get val probabilities
val_preds   = trainer.predict(val_dataset)
val_probs   = torch.softmax(torch.tensor(val_preds.predictions, dtype=torch.float32), dim=-1).numpy()[:, 1]
val_labels  = val_preds.label_ids

best_thresh, best_f1 = 0.5, 0.0
for t in np.arange(0.05, 0.60, 0.05):
    preds_t = (val_probs >= t).astype(int)
    score   = f1(val_labels, preds_t, zero_division=0)
    if score > best_f1:
        best_f1, best_thresh = score, t

print(f"Best threshold (val F1={best_f1:.3f}): {best_thresh:.2f}\n")

# Apply tuned threshold to test set
predictions = trainer.predict(test_dataset)
test_probs  = torch.softmax(torch.tensor(predictions.predictions, dtype=torch.float32), dim=-1).numpy()[:, 1]
pred_labels = (test_probs >= best_thresh).astype(int)
true_labels = predictions.label_ids

print("Classification Report:")
print(classification_report(true_labels, pred_labels, target_names=['NON-TOXIC', 'TOXIC']))

cm = confusion_matrix(true_labels, pred_labels)
print("Confusion Matrix (rows=actual, cols=predicted):")
print(f"            Pred NON-TOXIC  Pred TOXIC")
print(f"Actual NON:     {cm[0][0]:>6}        {cm[0][1]:>6}")
print(f"Actual TOX:     {cm[1][0]:>6}        {cm[1][1]:>6}")
print(f"\nToxic recall: {cm[1][1]}/{cm[1][0]+cm[1][1]} = {cm[1][1]/(cm[1][0]+cm[1][1]):.1%} compounds correctly flagged")

Best threshold (val F1=0.327): 0.55



Classification Report:
              precision    recall  f1-score   support

   NON-TOXIC       0.96      0.90      0.93       953
       TOXIC       0.26      0.50      0.34        64

    accuracy                           0.88      1017
   macro avg       0.61      0.70      0.64      1017
weighted avg       0.92      0.88      0.90      1017

Confusion Matrix (rows=actual, cols=predicted):
            Pred NON-TOXIC  Pred TOXIC
Actual NON:        862            91
Actual TOX:         32            32

Toxic recall: 32/64 = 50.0% compounds correctly flagged


---
## Section 7: Inference

This is the function you'd expose in a production pipeline or a Streamlit app. Given any SMILES string, it returns a prediction, confidence score, and raw probabilities.

Note: **always validate SMILES before running inference** in a real application. RDKit's `Chem.MolFromSmiles()` returns `None` for invalid SMILES — good to check.

In [33]:
def predict_toxicity(smiles: str, threshold: float = 0.5) -> dict:
    """
    Predict toxicity for a single SMILES string.

    Args:
        smiles:    A valid SMILES string representing a molecule.
        threshold: Decision threshold for TOXIC label (default 0.5).
                   Use `best_thresh` from the evaluation cell for calibrated output.

    Returns:
        dict with keys: smiles, prediction, confidence, prob_non_toxic, prob_toxic
    """
    model.eval()
    device = next(model.parameters()).device

    inputs = tokenizer(
        smiles,
        return_tensors="pt",
        padding='max_length',
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs    = torch.softmax(outputs.logits, dim=-1)[0].cpu()
    toxic_p  = probs[1].item()
    pred_idx = 1 if toxic_p >= threshold else 0

    return {
        "smiles":         smiles,
        "prediction":     "TOXIC" if pred_idx == 1 else "NON-TOXIC",
        "confidence":     round(probs[pred_idx].item(), 4),
        "prob_non_toxic": round(probs[0].item(), 4),
        "prob_toxic":     round(toxic_p, 4),
    }


# Test on well-known molecules using the tuned threshold
test_molecules = {
    "Aspirin":           "CC(=O)Oc1ccccc1C(=O)O",
    "Caffeine":          "Cn1c(=O)c2c(ncn2C)n(c1=O)C",
    "Ibuprofen":         "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "Benzene":           "c1ccccc1",
    "Ethanol":           "CCO",
    "Cisplatin (chemo)": "[Pt](Cl)(Cl)(N)N",
}

print(f"Predictions — SR-p53 toxicity  (threshold={best_thresh:.2f})\n")
print(f"{'Molecule':<22} {'Prediction':<12} {'P(toxic)':<10}")
print("-" * 44)
for name, smi in test_molecules.items():
    r = predict_toxicity(smi, threshold=best_thresh)
    print(f"{name:<22} {r['prediction']:<12} {r['prob_toxic']:<10.4f}")

Predictions — SR-p53 toxicity  (threshold=0.55)

Molecule               Prediction   P(toxic)  
--------------------------------------------
Aspirin                NON-TOXIC    0.0093    
Caffeine               NON-TOXIC    0.5122    
Ibuprofen              NON-TOXIC    0.0056    
Benzene                NON-TOXIC    0.0212    
Ethanol                NON-TOXIC    0.0214    
Cisplatin (chemo)      NON-TOXIC    0.1546    


---
## Section 8: Save, Push to Hub, and Log Reproducibility Metadata

Good AI governance means every model artifact is traceable back to:
- Which dataset version it was trained on
- Which hyperparameters were used
- What metrics it achieved
- When and by whom it was produced

We save this as a structured JSON file alongside the model.

In [34]:
# ── Save locally ──────────────────────────────────────────────────────────────
trainer.save_model(RUN_NAME)
tokenizer.save_pretrained(RUN_NAME)
print(f"✅ Model saved locally to: {RUN_NAME}/")

# ── Push to HuggingFace Hub ───────────────────────────────────────────────────
trainer.push_to_hub(
    commit_message=f"Fine-tuned ChemBERTa-77M on Tox21 {TARGET_TASK} "
                   f"| ROC-AUC: {test_results['eval_roc_auc']:.4f}"
)
print(f"✅ Model pushed to: https://huggingface.co/{HUB_MODEL_NAME}")

# ── Save reproducibility metadata ────────────────────────────────────────────
# This JSON file is the "passport" for this model artifact.
metadata = {
    "experiment": {
        "run_name":        RUN_NAME,
        "timestamp":       datetime.now().isoformat(),
        "hub_model":       HUB_MODEL_NAME,
    },
    "data": {
        "dataset":         DATASET_NAME,
        "target_task":     TARGET_TASK,
        "train_samples":   len(train_dataset),
        "val_samples":     len(val_dataset),
        "test_samples":    len(test_dataset),
        "positive_rate":   round(train_df['label'].mean(), 4),
        "seed":            SEED,
    },
    "model": {
        "base_model":      BASE_MODEL,
        "lora_r":          LORA_R,
        "lora_alpha":      LORA_ALPHA,
        "lora_dropout":    LORA_DROPOUT,
        "target_modules":  TARGET_MODULES,
    },
    "training": {
        "epochs":          EPOCHS,
        "batch_size":      BATCH_SIZE,
        "learning_rate":   LEARNING_RATE,
        "weight_decay":    WEIGHT_DECAY,
        "warmup_ratio":    WARMUP_RATIO,
        "max_seq_length":  MAX_SEQ_LENGTH,
        "optimizer":       "adamw_torch",
        "lr_scheduler":    "cosine",
    },
    "results": {
        "test_roc_auc":   test_results.get("eval_roc_auc"),
        "test_accuracy":  test_results.get("eval_accuracy"),
        "test_f1":        test_results.get("eval_f1"),
        "test_loss":      test_results.get("eval_loss"),
    },
}

metadata_path = f"{RUN_NAME}/experiment_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\n✅ Reproducibility metadata saved to: {metadata_path}")
print(json.dumps(metadata, indent=2))

Processing Files (2 / 2): 100%|██████████|  897kB /  897kB,   ???B/s  
New Data Upload: |          |  0.00B /  0.00B,   ???B/s  


✅ Model saved locally to: chemberta-tox21-sr-p53-20260519_0805/


Processing Files (2 / 2): 100%|██████████|  897kB /  897kB,   ???B/s  
New Data Upload: |          |  0.00B /  0.00B,   ???B/s  


✅ Model pushed to: https://huggingface.co/mike-malloy/chemberta-tox21-sr-p53-20260519_0805

✅ Reproducibility metadata saved to: chemberta-tox21-sr-p53-20260519_0805/experiment_metadata.json
{
  "experiment": {
    "run_name": "chemberta-tox21-sr-p53-20260519_0805",
    "timestamp": "2026-05-19T08:37:36.667599",
    "hub_model": "mike-malloy/chemberta-tox21-sr-p53-20260519_0805"
  },
  "data": {
    "dataset": "scikit-fingerprints/MoleculeNet_Tox21",
    "target_task": "SR-p53",
    "train_samples": 4741,
    "val_samples": 1016,
    "test_samples": 1017,
    "positive_rate": 0.0624,
    "seed": 42
  },
  "model": {
    "base_model": "DeepChem/ChemBERTa-77M-MTR",
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.1,
    "target_modules": [
      "query",
      "value"
    ]
  },
  "training": {
    "epochs": 10,
    "batch_size": 32,
    "learning_rate": 0.0002,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "max_seq_length": 128,
    "optimizer": "adamw_torch",
    

In [35]:
# ── Finish WandB run ──────────────────────────────────────────────────────────
wandb.finish()
print("✅ Training complete!")
print(f"   WandB dashboard: https://wandb.ai/{PROJECT_NAME}")
print(f"   HuggingFace Hub: https://huggingface.co/{HUB_MODEL_NAME}")

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


eval/accuracy,▇█████████████████████████▆▄▅▄▁▃▄▃▃▄▅▃▃▅
eval/f1,▄▃▁▁▁▁▁▁▂▂▂▃▃▃▃▄▃▄▃▄▄▃▄▄▄▂█▇▇▇▆▇▇▇▇▇█▇▇█
eval/loss,▇▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▆▆▇▇▆▇▇▇▇▇█▆▇▆
eval/roc_auc,▁▁▂▃▄▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▆▆▇▇▇▇▇▇▇▇▇▇▇█
eval/runtime,█▁▁▂▂▁▂▁▂▂▂▂▃▂▂▂▂▁▁▁▁▁▂▂▁▇▃▂▁▂▂▂▂▂▃▂▂▁▂▂
eval/samples_per_second,▁▇█▆▇▇▇█▆▇▇▇▅▇▇▇▇█████▇▇█▂▆▇█▇▇▆▇▆▆▇▇▇▇▆
eval/steps_per_second,▁▇█▆▇▇▇█▆▇▇▇▅▇▇▇▇█████▇▇█▂▆▇█▇▇▆▇▆▆▇▇▇▇▆
test/accuracy,██▂▁▂
test/f1,▁▁█▇█
test/loss,▁▁▇█▇
+9,...


✅ Training complete!
   WandB dashboard: https://wandb.ai/admet-toxicity
   HuggingFace Hub: https://huggingface.co/mike-malloy/chemberta-tox21-sr-p53-20260519_0805


---
## What's Next?

Once you've run this notebook successfully, the natural next milestones are:

**Milestone 2 — Multi-target prediction:**  
Extend this to predict all 12 Tox21 targets simultaneously using multi-label classification. This is more realistic for a production ADMET tool.

**Milestone 3 — Explainability:**  
Use attention weights or SHAP values to highlight which atoms/substructures the model is focusing on. This turns a black-box score into an interpretable signal.

**Milestone 4 — Molecular similarity retrieval:**  
Build a FAISS index over the training set embeddings so you can retrieve similar molecules to any query — useful for explaining predictions by analogy.

**Milestone 5 — Streamlit UI:**  
Wrap the `predict_toxicity()` function in a web app where users can draw molecules or paste SMILES and see live predictions.

---
*Project: ADMET Toxicity Prediction with ChemBERTa | Milestone 1*